# Milestone 1: Dataset Audit & Data Contract
## Adaptive Cyber Threat Intelligence under Concept Drift

This notebook performs a comprehensive quality audit of the cleaned **CICIDS2017** dataset (`cicids2017_cleaned.csv`).
It inspects dataset metrics, evaluates class imbalance, performs quality validation, and audits feature production feasibility to prevent **training-serving skew** when deploying live network sensors.

In [1]:
import sys
from pathlib import Path

# Add project root to sys.path
# Robust root resolution for CyberAdapt
cwd = Path.cwd().resolve()
while cwd.name and cwd.name != "CyberAdapt" and cwd.parent != cwd:
    cwd = cwd.parent
cyberadapt_root = cwd if cwd.name == "CyberAdapt" else Path("/home/tanishq/AI/CyberAdapt")
if str(cyberadapt_root) not in sys.path:
    sys.path.insert(0, str(cyberadapt_root))
ml_root = cyberadapt_root / "ml"
project_root = ml_root
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import IPython.display as display
from ml.src.data.audit import (
    inspect_dataset_meta,
    analyze_target_distribution,
    validate_data_quality,
    compute_numeric_stats,
    audit_feature_feasibility,
    plot_class_distribution,
    asdict
)

dataset_path = ml_root / "data" / "raw" / "cicids2017" / "cicids2017_cleaned.csv"
if not dataset_path.exists():
    dataset_path = ml_root / "data" / "raw" / "cicids2017_cleaned.csv"
output_dir = project_root / "artifacts" / "reports"
print(f"Project root: {project_root}")
print(f"Dataset path: {dataset_path}")

Project root: /home/tanishq/AI/CyberAdapt/ml
Dataset path: /home/tanishq/AI/CyberAdapt/ml/data/raw/cicids2017/cicids2017_cleaned.csv


--- 
### 1. Load Dataset & Inspect Metadata

In [2]:
df = pd.read_csv(dataset_path)
meta = inspect_dataset_meta(df, file_path=dataset_path, target_col="Attack Type")
print(f"Rows: {meta.num_rows:,}")
print(f"Columns: {meta.num_columns}")
print(f"Memory Usage: {meta.memory_usage_mb:.2f} MB")
print(f"File Size: {meta.file_size_mb:.2f} MB")
print(f"Target Present: {meta.target_column_present} ({meta.target_column_name})")

Rows: 2,520,751
Columns: 53
Memory Usage: 1049.55 MB
File Size: 684.02 MB
Target Present: True (Attack Type)


--- 
### 2. Target Class Distribution Analysis

In [3]:
target_df = analyze_target_distribution(df, target_col="Attack Type")
display.display(target_df)

# Display saved plot
plot_path = output_dir / "class_distribution.png"
if plot_path.exists():
    display.display(display.Image(filename=str(plot_path)))

,class,count,percentage,imbalance_ratio_vs_majority
0,Normal Traffic,2095057,83.1124,1.00
1,DoS,193745,7.6860,10.81
2,DDoS,128014,5.0784,16.37
3,Port Scanning,90694,3.5979,23.10
4,Brute Force,9150,0.3630,228.97
5,Web Attacks,2143,0.0850,977.63
6,Bots,1948,0.0773,1075.49


--- 
### 3. Data Quality Validation

In [4]:
quality_report = validate_data_quality(df)
print(f"Total Rows: {quality_report['total_rows']:,}")
print(f"Duplicate Rows: {quality_report['duplicate_rows']:,} ({quality_report['duplicate_row_percentage']:.2f}%)")
print(f"Missing Values: {quality_report['total_missing_values']}")
print(f"Columns with +Inf: {list(quality_report['columns_with_pos_inf'].keys())}")
print(f"Columns with -Inf: {list(quality_report['columns_with_neg_inf'].keys())}")
print(f"Constant Columns: {quality_report['constant_columns']}")
print(f"Low Cardinality Columns: {quality_report['low_cardinality_columns']}")

Total Rows: 2,520,751
Duplicate Rows: 161 (0.01%)
Missing Values: 0
Columns with +Inf: []
Columns with -Inf: []
Constant Columns: []
Low Cardinality Columns: ['FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count']


--- 
### 4. Feature Production Feasibility Assessment

In [5]:
feature_audit_df = audit_feature_feasibility(df, target_col="Attack Type")
display.display(feature_audit_df.head(15))

print("\nCategory breakdown:")
display.display(feature_audit_df['category'].value_counts())

,feature,dtype,category,reason,production_feasibility,notes
0,Destination Port,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
1,Flow Duration,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
2,Total Fwd Packets,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
3,Total Length of Fwd Packets,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
4,Fwd Packet Length Max,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
5,Fwd Packet Length Min,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
6,Fwd Packet Length Mean,float64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
7,Fwd Packet Length Std,float64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
8,Bwd Packet Length Max,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...
9,Bwd Packet Length Min,int64,A. LIVE-FEASIBLE,Basic per-packet or flow summary statistic ext...,HIGH,Can be computed by real-time ebpf/pcap flow se...



Category breakdown:


category
B. DERIVED-LIVE-FEASIBLE              24
A. LIVE-FEASIBLE                      23
C. DATASET-SPECIFIC / QUESTIONABLE     5
TARGET                                 1
Name: count, dtype: int64